# 02 — External Data Enrichment
**CMPE 188 | Flight Delay Prediction**

Goals:
- Build an IATA → lat/lon/elevation lookup from OurAirports (public domain CSV)
- Fetch annual climate normals per airport from OpenMeteo (free, no API key)
- Merge geographic + weather features onto the dataset as `from_*` / `to_*` columns
- Save enriched dataset to `data/processed/Airlines_enriched.csv`

*Derived features (time_bucket, is_peak_hour, route_volume, airline_delay_rate) are computed in `03_model_baseline.ipynb` since they require the train/test split.*

In [1]:
import time
import pandas as pd
import requests

DATA_RAW = "../data/raw/Airlines.csv"
DATA_WEATHER_CACHE = "../data/processed/airport_weather.csv"
DATA_OUT = "../data/processed/Airlines_enriched.csv"

df = pd.read_csv(DATA_RAW)
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
df.head()

Loaded 539,383 rows, 9 columns


,id,Airline,Flight,AirportFrom,AirportTo,DayOfWeek,Time,Length,Delay
0,1,CO,269,SFO,IAH,3,15,205,1
1,2,US,1558,PHX,CLT,3,15,222,1
2,3,AA,2400,LAX,DFW,3,20,165,1
3,4,AA,2466,SFO,DFW,3,20,195,1
4,5,AS,108,ANC,SEA,3,30,202,0


## 1. Airport Lookup Table (IATA → lat/lon/elevation)

In [2]:
OURAIRPORTS_URL = "https://davidmegginson.github.io/ourairports-data/airports.csv"

# All IATA codes present in the dataset
dataset_airports = sorted(set(df["AirportFrom"].unique()) | set(df["AirportTo"].unique()))
print(f"Unique airports in dataset: {len(dataset_airports)}")

# Download OurAirports — public domain, no key required
raw = pd.read_csv(OURAIRPORTS_URL, low_memory=False)

# Keep only airport entries with a valid IATA code
airports_raw = raw[
    raw["type"].isin(["large_airport", "medium_airport", "small_airport"])
    & raw["iata_code"].notna()
    & (raw["iata_code"] != "")
][["iata_code", "latitude_deg", "longitude_deg", "elevation_ft"]].copy()

airports_raw.columns = ["iata_code", "lat", "lon", "elevation_ft"]
airports_raw = airports_raw.drop_duplicates("iata_code").set_index("iata_code")

# Filter to only airports we actually need
airport_lookup = airports_raw[airports_raw.index.isin(dataset_airports)].copy()

missing = set(dataset_airports) - set(airport_lookup.index)
print(f"Matched: {len(airport_lookup)} / {len(dataset_airports)}  |  Missing: {len(missing)}")
if missing:
    print("Not found in OurAirports:", sorted(missing))

airport_lookup.head(10)

Unique airports in dataset: 293


Matched: 293 / 293  |  Missing: 0


,lat,lon,elevation_ft
iata_code,,,
ABE,40.651773,-75.442797,393.0
ABI,32.411301,-99.681900,1791.0
ABQ,35.039976,-106.608925,5355.0
ABR,45.449100,-98.421799,1302.0
ABY,31.532946,-84.196215,197.0
ACT,31.611300,-97.230499,516.0
ACV,40.978101,-124.109000,221.0
ACY,39.456201,-74.577511,75.0
AEX,31.325828,-92.546702,89.0


## 2. Weather Enrichment — OpenMeteo Climate Normals

In [3]:
import os

def fetch_weather(iata, lat, lon):
    """Fetch one year of daily climate data from OpenMeteo and return annual means."""
    url = "https://climate-api.open-meteo.com/v1/climate"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2019-01-01",
        "end_date": "2019-12-31",
        "models": "EC_Earth3P_HR",
        "daily": "temperature_2m_mean,precipitation_sum,wind_speed_10m_mean",
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()["daily"]
    return {
        "iata_code": iata,
        "avg_temperature":   pd.Series(data["temperature_2m_mean"]).mean(),
        "avg_precipitation": pd.Series(data["precipitation_sum"]).mean(),
        "avg_wind_speed":    pd.Series(data["wind_speed_10m_mean"]).mean(),
    }

# Load from cache if it exists, otherwise fetch and save
if os.path.exists(DATA_WEATHER_CACHE):
    weather_df = pd.read_csv(DATA_WEATHER_CACHE)
    print(f"Loaded weather cache: {len(weather_df)} airports")
else:
    print(f"Fetching weather for {len(airport_lookup)} airports (this runs once)...")
    records = []
    for i, (iata, row) in enumerate(airport_lookup.iterrows()):
        try:
            records.append(fetch_weather(iata, row["lat"], row["lon"]))
        except Exception as e:
            print(f"  [{i+1}/{len(airport_lookup)}] {iata} FAILED: {e}")
            continue
        if (i + 1) % 25 == 0:
            print(f"  {i+1}/{len(airport_lookup)} done...")
        time.sleep(0.1)

    weather_df = pd.DataFrame(records)
    weather_df.to_csv(DATA_WEATHER_CACHE, index=False)
    print(f"Done — cached {len(weather_df)} airports → {DATA_WEATHER_CACHE}")

weather_df.head()

Fetching weather for 293 airports (this runs once)...


  25/293 done...


  50/293 done...


  75/293 done...


  100/293 done...


...

  225/293 done...


  250/293 done...


  275/293 done...


Done — cached 293 airports → ../data/processed/airport_weather.csv


,iata_code,avg_temperature,avg_precipitation,avg_wind_speed
0,ABE,10.901644,3.446932,10.520548
1,ABI,17.791233,1.889699,16.836986
2,ABQ,13.581918,1.253562,9.352329
3,ABR,6.876986,1.815370,14.872055
4,ABY,19.417260,2.849151,10.476986


## 3. Merge Weather onto Dataset

In [4]:
# Build a full airport_weather table: geo + weather, indexed by IATA code
geo = airport_lookup.reset_index()  # cols: iata_code, lat, lon, elevation_ft
airport_weather = geo.merge(weather_df, on="iata_code", how="inner")
airport_weather = airport_weather.set_index("iata_code")

print(f"airport_weather shape: {airport_weather.shape}")
print(airport_weather.columns.tolist())
print(airport_weather.head())

# Merge onto main dataset: AirportFrom → from_* columns
weather_from = airport_weather.add_prefix("from_").reset_index().rename(columns={"iata_code": "AirportFrom"})
weather_to   = airport_weather.add_prefix("to_").reset_index().rename(columns={"iata_code": "AirportTo"})

df_enriched = df.merge(weather_from, on="AirportFrom", how="left")
df_enriched = df_enriched.merge(weather_to, on="AirportTo", how="left")

# Report coverage
weather_cols = [c for c in df_enriched.columns if c.startswith(("from_", "to_"))]
null_counts = df_enriched[weather_cols].isnull().sum()
total_rows = len(df_enriched)
print(f"\nRows with any missing weather: {df_enriched[weather_cols].isnull().any(axis=1).sum():,} / {total_rows:,}")
print(null_counts[null_counts > 0])

airport_weather shape: (293, 6)
['lat', 'lon', 'elevation_ft', 'avg_temperature', 'avg_precipitation', 'avg_wind_speed']
                 lat         lon  elevation_ft  avg_temperature  \
iata_code                                                         
ABE        40.651773  -75.442797         393.0        10.901644   
ABI        32.411301  -99.681900        1791.0        17.791233   
ABQ        35.039976 -106.608925        5355.0        13.581918   
ABR        45.449100  -98.421799        1302.0         6.876986   
ABY        31.532946  -84.196215         197.0        19.417260   

           avg_precipitation  avg_wind_speed  
iata_code                                     
ABE                 3.446932       10.520548  
ABI                 1.889699       16.836986  
ABQ                 1.253562        9.352329  
ABR                 1.815370       14.872055  
ABY                 2.849151       10.476986  

Rows with any missing weather: 0 / 539,383
Series([], dtype: int64)


## 4. Handle Missing Weather Values

Any rows whose airport wasn't matched will have nulls. Fill with the column global mean so no rows are dropped.

In [5]:
for col in weather_cols:
    n_null = df_enriched[col].isnull().sum()
    if n_null > 0:
        fill_val = df_enriched[col].mean()
        df_enriched[col] = df_enriched[col].fillna(fill_val)
        print(f"  Filled {n_null:,} nulls in {col} with mean={fill_val:.3f}")

print(f"\nNulls remaining: {df_enriched[weather_cols].isnull().sum().sum()}")
print(f"Shape after fill: {df_enriched.shape}")


Nulls remaining: 0
Shape after fill: (539383, 21)


## 5. Save Enriched Dataset

In [6]:
df_enriched.to_csv(DATA_OUT, index=False)

print(f"Saved → {DATA_OUT}")
print(f"Shape: {df_enriched.shape}")
print(f"\nColumns ({len(df_enriched.columns)}):")
for col in df_enriched.columns:
    print(f"  {col}")

Saved → ../data/processed/Airlines_enriched.csv
Shape: (539383, 21)

Columns (21):
  id
  Airline
  Flight
  AirportFrom
  AirportTo
  DayOfWeek
  Time
  Length
  Delay
  from_lat
  from_lon
  from_elevation_ft
  from_avg_temperature
  from_avg_precipitation
  from_avg_wind_speed
  to_lat
  to_lon
  to_elevation_ft
  to_avg_temperature
  to_avg_precipitation
  to_avg_wind_speed
